In [ ]:
import torch
import numpy 


class Dataloader:
    def __init__(self,batch_size,block_size,device):
        self.batch_size = batch_size
        self.block_size = block_size
        self.device = device

        self.train_data = np.memmamp("train.bin",dtype=np.uint16,mode="r")
        self.val_data = np.memmap("val.bin",dtype=np.uint16,mode="r")

    def forward(self,split="train"):

        d = self.train_data if split == "train" else self.val_data
        idx = torch.randn(0,len(d)-self.block_size,(self.batch_size,))

        x_list = [torch.from_numpy((d[i : i + self.block_size]).astype(np.int64)) for i in idx]
        y_list = [torch.from_numpy((d[i + 1 : i + 1 + self.block_size]).astype(np.int64)) for i in idx]

        x = torch.stack(x_list)
        y = torch.stack(y_list)

        if "cuda" in str(self.device):
            x = x.pin_memory().to(self.device,non_blocking=True)
            y = y.pin_memory().to(self.device,non_blocking=True)

        else:
            x,y = x.to(self.device),y.to(self.device)

        return x,y
            

# Dataset Class

So now we can create the class for the dataset. When it comes to the dataset class, we need to understand what it actually needs from us right?

At the dataset phase, we separate the data via **batch size**, **block size**, and then the **device**.

- **Batch size** — how many sequences run in parallel
- **Block size** — how many tokens I am gonna process at once
- **Device** — can be either CPU or GPU, so we must check this to not cause issues

Next we need to give the path of the files. Since I am doing it in the same folder, no worries — but you can check the main code for using `os.path.join()` methods if the files are elsewhere.

Now we are gonna use `np.memmap` — which is an efficient and advanced method to read data directly from the file on disk. We already saved `train.bin` and `val.bin` earlier, so now in the dataloader step we take `split` as a parameter.

We pass `split` for two reasons:
1. When the split is `train` — we target `train.bin` to compute the loss and update the weights
2. When the split is `val` — the model evaluates the loss to check if it's overfitting or not

Training data randomly samples chunks from the binary file continuously across iterations. Validation data evaluates fixed, repeatable slices so the evaluation loss curve doesn't bounce around due to random sample noise.

---

## The Index Bounds Problem

The core reason this works — we are gonna take random indices inside the bounds of the data.

Say the train data has indices from `0` to `200` and the block size is `6`. Can we start at index `199`? No, right — because it would go out of bounds.

The solution:

```
idx range = 0  to  len(data) - block_size - 1
```

The reason for that `-1` is because the `y` batch indices are `x + 1`. So if `x` ends at `200`, `y` would go to `201` which is out of bounds. So we use `len(d) - block_size - 1` for the `x` indices.

---

## Building the Batch

```python
x_list = [d[i : i + block_size] for i in idx]
```

`d` here is a NumPy `uint16` array and `idx` is a Torch tensor — so we need to convert. And before stacking, we cast `uint16` → `int64` because the internal C++ code of PyTorch expects 64-bit values.

```python
x = torch.stack(x_list)
```

`x_list` has individual single sequences. We use `torch.stack()` to convert those single pieces into a proper batch.

---

## Pinned Memory & Non-Blocking Transfer

Standard system RAM is **pageable** — the OS can move chunks of it around or swap it to disk if memory gets tight. The GPU cannot copy directly from pageable RAM via DMA.

```python
x = x.pin_memory()
x = x.to(device, non_blocking=True)
```

`pin_memory()` locks that specific tensor into physical host RAM so the OS cannot move or swap it — now the GPU can pull it directly from CPU RAM.

With `non_blocking=True` — the CPU tells the GPU's DMA engine to start copying the pinned memory and then immediately moves on to the next line of code. The copy happens in the background concurrently while the CPU continues working.
